# Exercise 1 · Give the error to Codex, not to a chatbot

You will let Codex run a failing script, read the error itself, and fix it. You copy nothing.

**Goal:** Ask Codex to run a script that fails, read its error and fix it, then check the fix before you keep it.

**Why it matters:** A coding agent can run the code and read the error itself. A screenshot in a chatbot shows only what you copied.

**Before you start:**
- Codex is installed and signed in (course README, step 1).
- You did the week 1 Codex exercises, or you know how to start Codex and leave it with `/exit`.

**What the check measures:** `timesheet.csv` is unchanged; `python hours.py` prints `Total hours: 37.5`; you committed the fix.

**Done when:** the last cell prints `Done:`.

## The situation

`hours.py` adds up the hours in a timesheet. It stops with an error. You do not look for the cause yourself: Codex finds it. The files are made up.

## Step 1 · Write the practice files

This is the solved copy. Run the cell. It writes `practice/` as it is after Step 6: `hours.py` with Codex's fix, the unchanged `timesheet.csv`, and two commits. It never overwrites a file.

Expect three `Wrote practice/` lines and `Committed the start and the fix`.

In [ ]:
FILES = {
    "hours.py": """\"\"\"Print the total hours in timesheet.csv.\"\"\"
import csv

total = 0.0
with open("timesheet.csv", newline="", encoding="utf-8") as sheet:
    for row in csv.DictReader(sheet):
        total += float(row["hours"].replace(",", "."))
print(f"Total hours: {total}")
""",
    "timesheet.csv": """day,hours
Monday,8
Tuesday,7.5
Wednesday,8
Thursday,"7,5"
Friday,6.5
""",
    ".gitignore": "__pycache__/\n",
}

# The fix as Codex may leave it after Step 4 (simulated): hours written with a comma are read as decimals.
import subprocess
from pathlib import Path

PRACTICE = Path("practice")
PRACTICE.mkdir(exist_ok=True)
for name, text in FILES.items():
    if (PRACTICE / name).exists():
        print(f"Kept practice/{name}: it is already there.")
    else:
        (PRACTICE / name).write_text(text, encoding="utf-8")
        print(f"Wrote practice/{name}")
if not (PRACTICE / ".git").exists():
    start = FILES["hours.py"].replace('row["hours"].replace(",", ".")', 'row["hours"]')
    (PRACTICE / "hours.py").write_text(start, encoding="utf-8")
    for step in (["init", "-q", "-b", "main"], ["add", "-A"], ["commit", "-qm", "Start"]):
        subprocess.run(["git", "-C", str(PRACTICE), *step], check=True)
    (PRACTICE / "hours.py").write_text(FILES["hours.py"], encoding="utf-8")
    subprocess.run(["git", "-C", str(PRACTICE), "commit", "-qam", "Read hours written with a comma"], check=True)
    print("Committed the start and the fix")

## Step 2 · Open a terminal and commit the start

Choose **Terminal > New Terminal**. Type each line, then press Enter:

```sh
source .venv/bin/activate
cd week-02/day-01/exercise-01-error-to-codex/practice
git init -q -b main && git add -A && git commit -qm "Start"
```

Expect no output from the last line. The commit lets you undo any change Codex makes.

## Step 3 · See the error once

```sh
python hours.py
```

Expect many lines. The last line starts with `ValueError: could not convert string to float`. Do not copy them. Codex reads them itself in Step 4.

### Learn: an agent runs the command and reads its output

A chatbot sees only the text you paste. Codex runs commands in your folder and reads what they print. So the request names the command, not the error. A good request also gives the goal, the limits and when the task is done ([Best practices](https://learn.chatgpt.com/codex/learn/best-practices), "Strong first use: Context and prompts").

## Step 4 · Give Codex the command, not the error

```sh
codex --no-daemon
```

`--no-daemon` runs Codex's commands in this terminal, with the course's Python. If Codex stops with an error about `--no-daemon`, run `codex` without it. If Codex asks "Trust this folder?", choose `1. Trust and continue`. Then type this request and press Enter:

```text
Run python hours.py. It fails. Find the cause and fix hours.py. Do not change timesheet.csv. Done when python hours.py prints the total of all five days.
```

Expect Codex to run the script, show the same error, and then edit `hours.py`.

## Step 5 · Read the fix, then keep it or reject it

Type `/diff` and read the change. Press `q` to close it. Then type `/exit`. Run the script yourself:

```sh
python hours.py
```

Keep the fix only if it prints `Total hours: 37.5`. A fix that skips the Thursday row prints `Total hours: 30.0`. Reject such a fix with `git restore hours.py`. Then go back to Step 4.

<details><summary>Show the answer</summary>

Keep a fix that reads `7,5` as 7.5, for example `float(row["hours"].replace(",", "."))`. Why: the five days add up to 8 + 7.5 + 8 + 7.5 + 6.5 = 37.5. A fix that skips a row hides the error and gives a wrong total.

</details>

## Step 6 · Commit the fix

```sh
git commit -qam "Read hours written with a comma"
git log --oneline
```

Expect two lines, newest first: your message, then `Start`.

## Start over (only if you need it)

Set `START_OVER = True`. Run the cell. It writes the Step 1 files again. Git keeps your commits. Then set it back to `False` and go to Step 2.

Expect `Nothing changed.` while it is `False`.

In [ ]:
# Start over: only if START_OVER is True, write the Step 1 files again
START_OVER = False

if not START_OVER:
    print("Nothing changed.")
elif "FILES" not in globals():
    print("Run the Step 1 cell first, then this cell.")
else:
    PRACTICE.mkdir(exist_ok=True)
    for name, text in FILES.items():
        (PRACTICE / name).write_text(text, encoding="utf-8")
    print("Wrote the practice files again. Now set START_OVER back to False.")

## Check your work

Run the cell. Expect three `ok:` lines, then `Done:`. A `Not yet:` line names the first problem and what to do.

In [ ]:
# Check your work: compare the timesheet, run hours.py, and read the Git log
import subprocess
import sys
from pathlib import Path

PRACTICE = Path("practice")
SHEET = "day,hours\nMonday,8\nTuesday,7.5\nWednesday,8\nThursday,\"7,5\"\nFriday,6.5\n"


def first_problem() -> str:
    """Print ok for each check that passes; return the first problem, or ""."""
    sheet = PRACTICE / "timesheet.csv"
    if not sheet.is_file():
        return "practice/timesheet.csv is missing. Run the Step 1 cell."
    if sheet.read_text(encoding="utf-8") != SHEET:
        return "timesheet.csv changed. Fix the code, not the data: run git restore timesheet.csv (Step 5)."
    print("ok: timesheet.csv is unchanged")
    run = subprocess.run([sys.executable, "hours.py"], cwd=PRACTICE, capture_output=True, text=True)
    said = (run.stdout.strip() or run.stderr.strip() or "nothing").splitlines()[-1]
    if said != "Total hours: 37.5":
        return f"python hours.py prints {said!r}, not 'Total hours: 37.5'. Give Codex the command (Step 4)."
    print("ok: python hours.py prints Total hours: 37.5")
    log = subprocess.run(["git", "-C", str(PRACTICE), "log", "--format=%s"], capture_output=True, text=True)
    if log.returncode or len(log.stdout.split("\n")) < 3:
        return "the fix is not committed. Commit it (Step 6)."
    changed = subprocess.run(["git", "-C", str(PRACTICE), "status", "--porcelain", "hours.py"],
                             capture_output=True, text=True).stdout.strip()
    if changed:
        return "hours.py has changes that are not committed. Commit them (Step 6)."
    print("ok: the fix is committed")
    return ""


problem = first_problem()
print(f"Not yet: {problem}" if problem else
      "Done: Codex found the error itself, and you kept a fix that gives the right total.")

## What you learned

- Codex runs a command and reads its error. You give it the command and the goal, not a screenshot.
- You still decide: read the diff, run the script, and keep the fix only if the result is right.

**Reflect:** Which error did you last paste into a chatbot? What would you type to Codex instead?